# Tratando e Analisando a base registrada de maneira manual com a de batidas na catraca

#### Importando bibliotecas essenciais para a Extração de Dados:

In [ ]:
# Importando bibliotecas:
import pandas as pd         # Manipula e analisa dados em tabelas...
from pathlib import Path    # Representa caminhos de arquivos e pastas...

#### Fazendo análise e trazendo um arquivo novo com colunas para representar o resultado:

In [ ]:
# ========= CAMINHOS =========
BASE_DIR = Path.cwd()

ARQ_HORAS = BASE_DIR / "xlsx" / "Horas Registradas.xlsx"

# Base AUTOMÁTICA (registros de catraca / ponto)
ARQ_COLAB = (
    BASE_DIR.parent
    / "Análise por Colaborador para Horas Extras"
    / "validation_base"
    / "parquet"
    / "Colaboradores.parquet"
)

ARQ_SAIDA = BASE_DIR / "Arquivo Solicitado - Colaboradores não Presentes.xlsx"

print("Horas Registradas (manual):", f"{ARQ_HORAS.parent.name}/{ARQ_HORAS.name}")
print("Base automática (Colaboradores):", f"{ARQ_COLAB.parent.name}/{ARQ_COLAB.name}")
print("Saída:", f"{ARQ_SAIDA.parent.name}/{ARQ_SAIDA.name}")

# ========= CONFIGURAÇÃO DAS COLUNAS =========
# Na base "Horas Registradas.xlsx" (MANUAL)
ID_COL_HORAS = "RG"                 # id/matrícula da pessoa
NOME_COL_HORAS = "Nome do empregado"  # nome da pessoa
DATA_COL_HORAS = "DATA"               # data em que deveria estar presente
HORA_MANUAL_COL = "HORA"              # total de horas em float

# Na base automática "Colaboradores.parquet"
ID_COL_VALID = "Cadastro"             # mesma identificação da pessoa
DATA_COL_VALID = "Data"               # data das marcações
HORA_AUTO_COL = "Hora"                # horário de cada marcação


# ========= LEITURA DA BASE HORAS (MANUAL) =========
df_horas = pd.read_excel(ARQ_HORAS)

print("Registros em Horas Registradas:", len(df_horas))

# Valida se a coluna HORA existe
if HORA_MANUAL_COL not in df_horas.columns:
    raise ValueError(
        f'Coluna "{HORA_MANUAL_COL}" não encontrada em Horas Registradas.xlsx. '
        f'Colunas disponíveis: {df_horas.columns.tolist()}'
    )

# Guarda a lista de colunas originais para manter no arquivo final
colunas_originais = df_horas.columns.tolist()

# ========= VALIDAÇÃO DAS COLUNAS EM HORAS =========
for col in [ID_COL_HORAS, NOME_COL_HORAS, DATA_COL_HORAS]:
    if col not in df_horas.columns:
        raise ValueError(
            f'Coluna "{col}" não encontrada em Horas Registradas.xlsx. '
            f'Colunas disponíveis: {df_horas.columns.tolist()}'
        )

# ========= CRIAR CHAVES AUXILIARES EM HORAS =========
df_horas["_ID_KEY"] = df_horas[ID_COL_HORAS].astype(str).str.strip()
df_horas["_DATA_KEY"] = pd.to_datetime(
    df_horas[DATA_COL_HORAS],
    dayfirst=True,
    errors="coerce"
).dt.date

# ========= LEITURA DA BASE AUTOMÁTICA (COLABORADORES) =========
df_colab = pd.read_parquet(ARQ_COLAB)

print("Registros na base automática (Colaboradores):", len(df_colab))

# Garante que as colunas necessárias existem
for col in [ID_COL_VALID, DATA_COL_VALID, HORA_AUTO_COL]:
    if col not in df_colab.columns:
        raise ValueError(
            f'Coluna "{col}" não encontrada em Colaboradores.parquet. '
            f'Colunas disponíveis: {df_colab.columns.tolist()}'
        )

# ========= CHAVES AUXILIARES NA BASE AUTOMÁTICA =========
df_colab["_ID_KEY"] = df_colab[ID_COL_VALID].astype(str).str.strip()
df_colab["_DATA_KEY"] = pd.to_datetime(
    df_colab[DATA_COL_VALID],
    dayfirst=True,
    errors="coerce"
).dt.date

# Datetime completo da marcação automática (Data + Hora)
df_colab["dt_marca"] = pd.to_datetime(
    df_colab[DATA_COL_VALID].astype(str).str.strip() + " " +
    df_colab[HORA_AUTO_COL].astype(str).str.strip(),
    dayfirst=True,
    errors="coerce"
)

# Mantém só linhas válidas
df_colab = df_colab.dropna(subset=["dt_marca", "_ID_KEY", "_DATA_KEY"])

# ========= DEFINIR PRESENÇA (apenas se tem pelo menos uma marcação automática no dia) =========
chaves_valid = df_colab[["_ID_KEY", "_DATA_KEY"]].drop_duplicates()

# Merge para marcar presença
df_merge = df_horas.merge(
    chaves_valid,
    on=["_ID_KEY", "_DATA_KEY"],
    how="left",
    indicator=True
)

# Máscara de presença
presenca_mask = df_merge["_merge"].eq("both")

print("Total de registros com Presença = True:", presenca_mask.sum())
print("Total de registros com Presença = False:", (~presenca_mask).sum())

# ========= CÁLCULO DAS HORAS AUTOMÁTICAS (primeira e última marcação do dia) =========
agg_horas = (
    df_colab
    .groupby(["_ID_KEY", "_DATA_KEY"])["dt_marca"]
    .agg(["min", "max"])  # primeira e última marcação
    .reset_index()
)

agg_horas.rename(
    columns={
        "min": "Primeira_marcacao",
        "max": "Ultima_marcacao"
    },
    inplace=True
)

# intervalo em horas (automático)
agg_horas["Horas_automaticas"] = (
    (agg_horas["Ultima_marcacao"] - agg_horas["Primeira_marcacao"])
    .dt.total_seconds() / 3600.0
)

# Junta essas informações no df_merge
df_merge = df_merge.merge(
    agg_horas,
    on=["_ID_KEY", "_DATA_KEY"],
    how="left"
)

# ========= COMPARAR HORAS AUTOMÁTICAS x HORAS MANUAIS =========
# Converte HORA (manual) para float
df_merge["HORA_float"] = pd.to_numeric(df_merge["HORA"], errors="coerce")

# Nova coluna: se trabalhou todo o período (somente para quem teve presença)
df_merge["Trabalhou_todo_periodo"] = pd.NA

cond = (
    presenca_mask
    & df_merge["Horas_automaticas"].notna()
    & df_merge["HORA_float"].notna()
)

# Aqui você pode escolher:
#   ==  -> tem que bater igual (considerando arredondamento)
#   >=  -> automático cobre pelo menos o que está no manual
df_merge.loc[cond, "Trabalhou_todo_periodo"] = (
    df_merge.loc[cond, "Horas_automaticas"].round(2)
    >= df_merge.loc[cond, "HORA_float"].round(2)
)

# ========= MONTAR DATAFRAME FINAL =========
df_final = df_merge[colunas_originais].copy()

# Coluna de presença (base automática)
df_final["Presença"] = presenca_mask

# Informações da base automática para você conferir
df_final["Primeira_marcacao_auto"] = df_merge["Primeira_marcacao"]
df_final["Ultima_marcacao_auto"]   = df_merge["Ultima_marcacao"]
df_final["Horas_automaticas"]      = df_merge["Horas_automaticas"]

# Coluna que responde à sua pergunta:
# "se a pessoa trabalhou por todo aquele período" (automático x manual)
df_final["Trabalhou_todo_periodo"] = df_merge["Trabalhou_todo_periodo"]

# ========= SALVAR SAÍDA =========
df_final.to_excel(ARQ_SAIDA, index=False)
print("Arquivo gerado com sucesso em:")
print(f"{ARQ_SAIDA.parent.name}/{ARQ_SAIDA.name}")
